# 03 - Özellik Çıkarımı: LBP + GLCM

Her 256×256 görüntüden iki tür doku özelliği çıkarıyoruz:

| Yöntem | Ayar | Boyut |
|---|---|---|
| **LBP** (Yerel İkili Örüntüler) | P=8, R=1, `nri_uniform` | 59 |
| **GLCM** (Gri Seviye Eş Oluşum Matrisi) | d=1; 0°, 45°, 90°, 135°; 6 özellik | 24 |
| **Toplam** | | **83** |

Önce `02` notebook'u çalıştırılmış ve `data/processed/images_256/` klasörü dolu olmalı.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src import features as F
from src.preprocess import read_gray, PROCESSED_IMG_DIR
sns.set_theme(style="whitegrid")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

idx = pd.read_csv(PROCESSED_IMG_DIR / "index.csv")
print(len(idx), "görüntü,", idx["family"].nunique(), "aile")

## 1. LBP: tek görüntü üzerinde

LBP her pikseli 8 komşusuyla karşılaştırır. Komşu merkezden büyük ya da eşitse 1, küçükse 0 yazılır. Çıkan 8 bitlik örüntü pikselin kodu olur. `nri_uniform` yönteminde en fazla 2 geçiş (0→1 veya 1→0) içeren 58 "düzgün" örüntü ayrı ayrı sayılır. Geri kalan bütün örüntüler tek bir kutuda toplanır, böylece 59 kutu elde edilir.

In [ ]:
ornek_aileler = idx["family"].value_counts().index[:3].tolist()
fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for i, fam in enumerate(ornek_aileler):
    img = read_gray(idx[idx["family"] == fam]["path"].iloc[0])
    axes[i, 0].imshow(img, cmap="gray"); axes[i, 0].set_title(f"{fam}"); axes[i, 0].axis("off")
    axes[i, 1].imshow(F.lbp_image(img), cmap="nipy_spectral"); axes[i, 1].set_title("LBP kod görüntüsü"); axes[i, 1].axis("off")
    axes[i, 2].bar(range(F.N_LBP_BINS), F.lbp_features(img), color="steelblue")
    axes[i, 2].set_title("LBP histogramı (59)"); axes[i, 2].set_xlabel("örüntü no")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "03_lbp_ornek.png", dpi=150)
plt.show()

## 2. GLCM: tek görüntü üzerinde

GLCM, belirli bir yönde yan yana duran iki pikselin (i, j) gri değer çiftinin ne sıklıkla görüldüğünü sayar. Bu matristen 6 özellik hesaplanır: kontrast, farklılık, homojenlik, enerji, korelasyon ve ASM. Her biri 4 açı için ayrı ayrı hesaplandığından 24 değer çıkar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, fam in zip(axes, ornek_aileler):
    img = read_gray(idx[idx["family"] == fam]["path"].iloc[0])
    m = F.glcm_matrix(img)[:, :, 0, 0]           # d=1, 0°
    ax.imshow(np.log1p(m * 1e4), cmap="magma", origin="lower")
    ax.set_title(f"{fam} - GLCM (0°, log ölçek)"); ax.set_xlabel("j"); ax.set_ylabel("i")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "03_glcm_ornek.png", dpi=150)
plt.show()

img = read_gray(idx["path"].iloc[0])
g = pd.Series(F.glcm_features(img), index=F.glcm_feature_names())
pd.DataFrame(g.values.reshape(len(config.GLCM_PROPS), -1),
             index=config.GLCM_PROPS, columns=[f"{a}°" for a in config.GLCM_ANGLES_DEG]).round(4)

## 3. Tüm veri seti için özellik çıkarımı

Bir görüntü yaklaşık 50 ms sürüyor, tek çekirdekte 9.339 görüntü 8 dakika civarında tamamlanır. `n_jobs=-1` ile bilgisayarın bütün çekirdekleri kullanılır.

> Windows'ta `n_jobs=-1` notebook içinde sorun çıkarırsa terminalden `python -m src.features --n-jobs -1` komutunu çalıştırın.

In [ ]:
feats = F.extract_dataset(n_jobs=-1)
feats.head()

In [ ]:
X, y, meta = F.load_features()
print("X:", X.shape, "| y:", y.shape)
print("Eksik değer:", int(X.isna().sum().sum()), "| Sonsuz değer:", int(np.isinf(X.values).sum()))
X.describe().T.head(10)

## 4. Özellikler aileleri ayırt ediyor mu?

Bir özelliğin aile ortalamaları birbirinden belirgin şekilde farklıysa, o özellik sınıflandırmada işe yarayacak demektir.

In [ ]:
secili = ["glcm_contrast_1_0", "glcm_homogeneity_1_0", "glcm_energy_1_0", "glcm_correlation_1_0"]
sira = y.value_counts().index
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, col in zip(axes.flat, secili):
    sns.boxplot(x=X[col], y=y, order=sira, ax=ax, fliersize=1)
    ax.set_title(col); ax.set_ylabel(""); ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "03_glcm_aile_kutu.png", dpi=150)
plt.show()

In [ ]:
# Aile başına ortalama LBP histogramı (ısı haritası)
lbp_cols = F.lbp_feature_names()
ort = X[lbp_cols].groupby(y).mean().loc[sira]
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(np.log10(ort + 1e-6), cmap="viridis", ax=ax, cbar_kws={"label": "log10(oran)"})
ax.set_xlabel("LBP örüntü no"); ax.set_ylabel("")
ax.set_title("Aile başına ortalama LBP histogramı")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "03_lbp_aile_isi.png", dpi=150)
plt.show()

## 5. GLCM özellikleri arasındaki korelasyon

Çok yüksek korelasyonlu (|r| > 0.95) özellikler aynı bilgiyi taşır. Bunu raporda belirtmek, 5. haftadaki modellerin sonuçlarını yorumlarken işe yarar.

In [ ]:
glcm_cols = F.glcm_feature_names()
corr = X[glcm_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, xticklabels=True, yticklabels=True)
ax.tick_params(labelsize=6)
ax.set_title("GLCM özellikleri korelasyon matrisi")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "03_glcm_korelasyon.png", dpi=150)
plt.show()

yuksek = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(glcm_cols)
          for b in glcm_cols[i + 1:] if abs(corr.loc[a, b]) > 0.95]
print(f"|r| > 0.95 olan çift sayısı: {len(yuksek)}")
pd.DataFrame(yuksek, columns=["özellik 1", "özellik 2", "r"]).head(15)

## 6. Notlar

- Özellik çıkarımı süresi (tek çekirdek / paralel):
- Aileleri en iyi ayırıyor gibi görünen GLCM özellikleri:
- LBP ısı haritasında birbirine çok benzeyen aileler:
- Yüksek korelasyonlu özellik grupları: